# TranSTR + DINOv3 + GroundingDINO-SigLIP2 + Top-5 Temporal Relation + NCOD LUM+HUM — Colab Pro
**Object feature mới**: SigLIP2 ROI 768D + DeBERTa class 768D + bbox 4D = 1540D. Data tải từ Kaggle; bs=32, không gradient accumulation, 20 epochs, EMA weights cho eval/save.


In [ ]:
# CELL 1: Git Clone & Setup (Google Colab)
import os
from pathlib import Path

REPO_URL = 'https://github.com/DanielQH07/tranSTR_Casual.git'
REPO_NAME = 'tranSTR_Casual'
BRANCH = 'feat/generic-train-improvements'
WORKING_DIR = Path('/content')
REPO_DIR = WORKING_DIR / REPO_NAME

def has_repo_files(p):
    p = Path(p)
    return (p / 'DataLoader.py').exists() and (p / 'networks' / 'model.py').exists()

if not has_repo_files(Path.cwd()):
    if not REPO_DIR.exists():
        rc = os.system(f'git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}')
        if rc != 0:
            raise RuntimeError('git clone failed')
    target_dir = REPO_DIR / 'causalvid'
    os.chdir(target_dir if target_dir.exists() else REPO_DIR)
if not has_repo_files(Path.cwd()):
    raise FileNotFoundError(f'Repo files not found in {Path.cwd()}')
print(f'Working directory: {Path.cwd()}')


In [ ]:
%cd /content/tranSTR_Casual

In [ ]:
# CELL 2: Dependencies + W&B login (Colab)
print('=== CELL 2: Dependencies & W&B Setup ===')
import os, sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'wandb', 'kagglehub', 'transformers', 'sentencepiece'])
import wandb

WANDB_PROJECT = 'transtr-causalvid-dino'
WANDB_ENTITY = None
wandb_key = ""
try:
    from google.colab import userdata
    wandb_key = wandb_key or (userdata.get('WANDB_API_KEY') or '').strip()
except Exception:
    pass
if wandb_key:
    wandb.login(key=wandb_key)
    print('W&B login OK')
else:
    os.environ.setdefault('WANDB_MODE', 'offline')
    print('WANDB_API_KEY is not configured; W&B offline mode enabled')


In [ ]:
# CELL 3: Resolve data paths from KaggleHub (Colab)
print('=== CELL 3: Data Paths (KaggleHub) ===')
import os, shutil
from pathlib import Path
import kagglehub

def _download(slug, env_name=None):
    override = os.environ.get(env_name, '').strip() if env_name else ''
    return Path(override) if override and Path(override).exists() else Path(kagglehub.dataset_download(slug))
def _find_dir_with_ext(root, ext):
    counts = {}
    for p in Path(root).rglob('*' + ext):
        counts[str(p.parent)] = counts.get(str(p.parent), 0) + 1
    return Path(max(counts, key=counts.get)) if counts else None
def _find_named(root, name):
    root = Path(root)
    if root.name.lower() == name.lower():
        return root
    return next((p for p in root.rglob('*') if p.is_dir() and p.name.lower() == name.lower()), None)

dinov3_root = _download('danielq07/dinov3-feat', 'DINOV3_DATASET_PATH')
gdino_root = _download(os.environ.get('GDINO_DATASET', 'danielq07/causal-vidqa-gdinofasterrcnn-features-merged'), 'GDINO_DATASET_PATH')
anno_root = _download('lusnaw/text-annotation', 'ANNO_DATASET_PATH')
split_root = _download('danielq07/casual-vid-data-split', 'SPLIT_DATASET_PATH')

all_dinov3_pt = list(Path(dinov3_root).rglob('*.pt'))
if not all_dinov3_pt:
    raise FileNotFoundError(f'No DINOv3 .pt files found under {dinov3_root}')

GDINO_MERGED_PATH = _find_dir_with_ext(gdino_root, '.pkl') or gdino_root
ANNOTATION_QA_PATH = _find_named(anno_root, 'QA') or anno_root
SPLIT_TXT_PATH = _find_named(split_root, 'split') or split_root

# DataLoader expects one flat directory. KaggleHub/Colab cache keeps DINOv3
# under train/valid/test, so link all three splits into a single view.
CLIP_MERGED_PATH = Path('/content/dinov3_T16_dim1024_merge')
CLIP_MERGED_PATH.mkdir(parents=True, exist_ok=True)
for src in all_dinov3_pt:
    dst = CLIP_MERGED_PATH / src.name
    if dst.exists():
        continue
    try:
        dst.symlink_to(src)
    except Exception:
        shutil.copy2(src, dst)

merged_count = len(list(CLIP_MERGED_PATH.glob('*.pt')))
source_unique_count = len({src.name for src in all_dinov3_pt})
if merged_count != source_unique_count:
    raise RuntimeError(
        f'DINOv3 merge incomplete: merged={merged_count}, source_unique={source_unique_count}'
    )
print(f'DINOv3 merged all splits: {merged_count} files')
print('Resolved paths:', CLIP_MERGED_PATH, GDINO_MERGED_PATH, ANNOTATION_QA_PATH, SPLIT_TXT_PATH)
for name, path in [('DINOv3', CLIP_MERGED_PATH), ('GDINO+FRCNN', GDINO_MERGED_PATH), ('QA', ANNOTATION_QA_PATH), ('split', SPLIT_TXT_PATH)]:
    if not Path(path).exists(): raise FileNotFoundError(f'{name} path not found: {path}')


In [ ]:
# CELL 4: Core imports + NCOD/HUM + Verifier/Knowledge training functions
print('=== CELL 4: Imports + Functions (NCOD + HUM + Verifier/Knowledge) ===')

import json
import math
import copy
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm

from utils.util import set_seed, set_gpu_devices
from DataLoader import VideoQADataset
from networks.model import VideoQAmodel

def _unpack_batch(batch):
    if len(batch) == 7:
        ff, of, q, a, ans_id, qns_key, q_family_id = batch
    elif len(batch) == 6:
        ff, of, q, a, ans_id, qns_key = batch
        q_family_id = None
    else:
        raise ValueError(f'Unexpected batch format with {len(batch)} elements')
    return ff, of, q, a, ans_id, qns_key, q_family_id

def _compute_sample_indices(qns_keys, key_to_idx, device):
    idxs = [key_to_idx.get(str(k), -1) for k in qns_keys]
    if any(i < 0 for i in idxs):
        missing = [str(qns_keys[i]) for i, v in enumerate(idxs) if v < 0][:5]
        raise KeyError(f'Missing qns_key in key_to_idx mapping: {missing}')
    return torch.tensor(idxs, dtype=torch.long, device=device)

_QTYPE_SUFFIXES = [
    'counterfactual_reason',
    'predictive_reason',
    'counterfactual',
    'predictive',
    'explanatory',
    'descriptive',
]

def _split_qns_key(qns_key):
    key = str(qns_key)
    for qtype in _QTYPE_SUFFIXES:
        suffix = f'_{qtype}'
        if key.endswith(suffix):
            return key[:-len(suffix)], qtype
    return key, 'unknown'

def _compute_acc_all_metrics(type_results):
    mapping = [
        ('Description', 'descriptive'),
        ('Explanation', 'explanatory'),
        ('Predictive-Answer', 'predictive'),
        ('Predictive-Reason', 'predictive_reason'),
        ('Counterfactual-Answer', 'counterfactual'),
        ('Counterfactual-Reason', 'counterfactual_reason'),
    ]
    metrics = {}
    for name, qtype in mapping:
        rows = type_results.get(qtype, [])
        metrics[name] = (sum(1 for r in rows if r['correct']) / len(rows) * 100) if rows else 0.0

    def _hard_metric(type_ans, type_reason):
        ans_by_vid = {r['video_id']: r['correct'] for r in type_results.get(type_ans, [])}
        reason_by_vid = {r['video_id']: r['correct'] for r in type_results.get(type_reason, [])}
        common_vids = set(ans_by_vid) & set(reason_by_vid)
        if not common_vids:
            return 0.0
        both_correct = sum(1 for vid in common_vids if ans_by_vid[vid] and reason_by_vid[vid])
        return both_correct / len(common_vids) * 100

    metrics['PAR'] = _hard_metric('predictive', 'predictive_reason')
    metrics['CAR'] = _hard_metric('counterfactual', 'counterfactual_reason')
    metrics['Acc_ALL'] = (metrics['Description'] + metrics['Explanation'] + metrics['PAR'] + metrics['CAR']) / 4.0
    return metrics

def _hard_negative_weights(cand_feat, target, enabled=False, max_weight=1.5):
    if (not enabled) or cand_feat is None or max_weight <= 1.0:
        return torch.ones_like(target, dtype=torch.float32)
    with torch.no_grad():
        cand_norm = F.normalize(cand_feat.detach(), dim=-1)
        gold = cand_norm.gather(1, target.view(-1, 1, 1).expand(-1, 1, cand_norm.size(-1))).squeeze(1)
        sims = torch.bmm(cand_norm, gold.unsqueeze(-1)).squeeze(-1)
        sims.scatter_(1, target.view(-1, 1), -1.0)
        hardness = sims.max(dim=1).values.clamp(min=0.0, max=1.0)
        return 1.0 + (float(max_weight) - 1.0) * hardness

def _update_ema_model(ema_model, model, decay):
    if ema_model is None:
        return
    with torch.no_grad():
        for ema_param, param in zip(ema_model.parameters(), model.parameters()):
            ema_param.data.mul_(decay).add_(param.data, alpha=1.0 - decay)
        for ema_buffer, buffer in zip(ema_model.buffers(), model.buffers()):
            ema_buffer.copy_(buffer)

def train_epoch_integrated(
    model, optimizer_model, optimizer_u, U, loader, xe, bce, device, epoch, key_to_idx,
    accumulation_steps=4, hum_alpha=1.0, lambda_verifier=0.2, lambda_knowledge=0.1,
    use_hard_neg_mining=False, hard_neg_weight_max=1.5,
    ema_model=None, ema_decay=0.999,
    scheduler=None, scheduler_step_per_batch=False
):
    model.train()
    total_loss, total_l1, total_l2 = 0.0, 0.0, 0.0
    total_verifier, total_knowledge = 0.0, 0.0
    correct, total = 0, 0
    optimizer_model.zero_grad()
    optimizer_u.zero_grad()

    pbar = tqdm(loader, desc=f'Epoch {epoch}', leave=False)
    for batch_idx, batch in enumerate(pbar):
        ff, of, q, a, ans_id, qns_keys, q_family_id = _unpack_batch(batch)
        ff, of, tgt = ff.to(device), of.to(device), ans_id.to(device)

        if q_family_id is None:
            q_family_id = torch.zeros_like(tgt)
        else:
            q_family_id = q_family_id.to(device)

        sample_indices = _compute_sample_indices(qns_keys, key_to_idx, device)
        out = model(ff, of, q, a, return_aux=True, q_family_id=q_family_id)
        logits = out['logits']
        fused_logits = out.get('fused_score', logits)
        verifier_logits = out.get('verifier_logits', logits)
        knowledge_logits = out.get('knowledge_score', None)
        cand_feat = out.get('cand_feat', None)

        probs = torch.softmax(logits, dim=1)
        y_onehot = F.one_hot(tgt, num_classes=logits.size(-1)).float()
        u_batch = U[sample_indices].unsqueeze(1)

        ce_per_sample = -torch.sum(y_onehot * torch.log(torch.clamp(probs, min=1e-12)), dim=1)
        shifted_probs = torch.clamp(probs + (u_batch.detach() * y_onehot), min=1e-12, max=1.0)
        lum_loss = -torch.sum(y_onehot * torch.log(shifted_probs), dim=1)
        hum_loss = (1.0 + hum_alpha * u_batch.detach().squeeze(1)) * ce_per_sample

        hard_neg_w = _hard_negative_weights(
            cand_feat,
            tgt,
            enabled=use_hard_neg_mining,
            max_weight=hard_neg_weight_max,
        )
        hard_mask = (q_family_id >= 2)
        l1_per_sample = torch.where(hard_mask, hum_loss, lum_loss)
        l1 = (l1_per_sample * hard_neg_w).mean()

        verifier_loss = bce(verifier_logits, y_onehot)
        if lambda_knowledge > 0.0 and knowledge_logits is not None:
            knowledge_loss = bce(knowledge_logits, y_onehot)
        else:
            knowledge_loss = torch.tensor(0.0, device=device)

        model_loss = l1 + lambda_verifier * verifier_loss + lambda_knowledge * knowledge_loss
        (model_loss / accumulation_steps).backward()

        shifted_det = probs.detach() + (u_batch * y_onehot)
        l2 = F.mse_loss(shifted_det, y_onehot)
        (l2 / accumulation_steps).backward()

        if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer_model.step()
            _update_ema_model(ema_model, model, ema_decay)
            if scheduler is not None and scheduler_step_per_batch:
                scheduler.step()
            optimizer_model.zero_grad()
            optimizer_u.step()
            optimizer_u.zero_grad()
            with torch.no_grad():
                U.clamp_(0.0, 0.99)

        total_l1 += l1.item()
        total_l2 += l2.item()
        total_verifier += verifier_loss.item()
        total_knowledge += knowledge_loss.item()
        total_loss += (model_loss + l2).item()
        correct += (fused_logits.argmax(-1) == tgt).sum().item()
        total += tgt.size(0)

        pbar.set_postfix({
            'loss': total_loss / (batch_idx + 1),
            'l1': total_l1 / (batch_idx + 1),
            'l2': total_l2 / (batch_idx + 1),
            'ver': total_verifier / (batch_idx + 1),
            'know': total_knowledge / (batch_idx + 1),
            'acc': correct / max(total, 1) * 100
        })

    n = len(loader)
    return (
        total_loss / n,
        total_l1 / n,
        total_l2 / n,
        total_verifier / n,
        total_knowledge / n,
        correct / max(total, 1) * 100
    )

def eval_epoch(model, loader, device):
    model.eval()
    correct, total = 0, 0
    type_results = {}
    with torch.no_grad():
        for batch in loader:
            ff, of, q, a, ans_id, qns_keys, q_family_id = _unpack_batch(batch)
            ff, of, tgt = ff.to(device), of.to(device), ans_id.to(device)
            q_family_id = q_family_id.to(device) if q_family_id is not None else None
            out = model(ff, of, q, a, return_aux=True, q_family_id=q_family_id)
            logits = out.get('fused_score', out['logits'])
            preds = logits.argmax(-1)
            correct += (preds == tgt).sum().item()
            total += tgt.size(0)

            for key, pred, target in zip(qns_keys, preds.detach().cpu().tolist(), tgt.detach().cpu().tolist()):
                video_id, qtype = _split_qns_key(key)
                type_results.setdefault(qtype, []).append({
                    'video_id': video_id,
                    'correct': bool(int(pred) == int(target)),
                })

    metrics = _compute_acc_all_metrics(type_results)
    metrics['Plain_Acc'] = correct / max(total, 1) * 100
    return metrics
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('Imports and functions defined for integrated training.')

In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('Imports OK')

In [ ]:
# CELL 5: Setup Paths & Config (Kaggle, bs=8 + accum=4 -> effective bs=32)
print('=== CELL 5: Paths & Config ===')

clip_merged_path   = globals().get('CLIP_MERGED_PATH', None)
gdino_merged_path  = globals().get('GDINO_MERGED_PATH', None)
annotation_qa_path = globals().get('ANNOTATION_QA_PATH', None)
split_txt_path     = globals().get('SPLIT_TXT_PATH', None)

CLIP_FEATURE_PATH  = clip_merged_path  or '/content/dinov3_T16_dim1024_merge'
GDINO_FEATURE_PATH = gdino_merged_path or str(GDINO_MERGED_PATH)
ANNOTATION_PATH    = annotation_qa_path or str(ANNOTATION_QA_PATH)
SPLIT_DIR          = split_txt_path    or str(SPLIT_TXT_PATH)

BASE = '/content/working' if os.path.exists('/content') else os.getcwd()
MODEL_DIR = os.path.join(BASE, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

print('\n--- Path Verification ---')
def verify_path(name, path):
    if os.path.exists(path):
        items = os.listdir(path)[:3]
        print(f'OK {name}: {items}')
        return True
    print(f'NOT FOUND {name}: {path}')
    return False

all_ok = True
all_ok &= verify_path('DINOv3 Features (1024D)', CLIP_FEATURE_PATH)
all_ok &= verify_path('GDINO+FRCNN Features (2820D)', GDINO_FEATURE_PATH)
all_ok &= verify_path('Annotations (QA)', ANNOTATION_PATH)
all_ok &= verify_path('Splits', SPLIT_DIR)

if not all_ok:
    raise FileNotFoundError('One or more required data paths are missing. Re-run CELL 3 or attach Kaggle datasets.')

import glob as _glob
n_pt  = len(_glob.glob(os.path.join(CLIP_FEATURE_PATH, '*.pt')))
n_pkl = len(_glob.glob(os.path.join(GDINO_FEATURE_PATH, '*.pkl')))
print(f'\nDINOv3 .pt: {n_pt} | GDINO+FRCNN .pkl: {n_pkl}')

# ============================================
# 3-RUN TUNING PRESETS
# ============================================
RUN_TRAINING = True
RUN_PROFILE = 'run1'
RUN_VARIANT = 'generic_safe_hn_ema_cos_nokb'
RUN3_REG_MODE = 'dropout'

RUN_PROFILES = {
    'baseline': {
        'epoch': 10, 'lr': 1e-5,
        'lambda_verifier': 0.3, 'lambda_knowledge': 0.0,
        'early_stop_start_epoch': 5, 'early_stop_patience': 4,
        'dropout': 0.3, 'encoder_dropout': 0.3, 'decay': 1e-4,
    },
    'run1': {
        'epoch': 10, 'lr': 1e-5,
        'lambda_verifier': 0.25, 'lambda_knowledge': 0.0,
        'early_stop_start_epoch': 5, 'early_stop_patience': 4,
        'dropout': 0.3, 'encoder_dropout': 0.3, 'decay': 1e-4,
    },
    'run2': {
        'epoch': 10, 'lr': 8e-6,
        'lambda_verifier': 0.25, 'lambda_knowledge': 0.0,
        'early_stop_start_epoch': 6, 'early_stop_patience': 5,
        'dropout': 0.3, 'encoder_dropout': 0.3, 'decay': 1e-4,
    },
    'run3': {
        'epoch': 10, 'lr': 8e-6,
        'lambda_verifier': 0.25, 'lambda_knowledge': 0.0,
        'early_stop_start_epoch': 6, 'early_stop_patience': 5,
        'dropout': 0.3, 'encoder_dropout': 0.3, 'decay': 1e-4,
    },
}

if RUN_PROFILE not in RUN_PROFILES:
    raise ValueError(f'Invalid RUN_PROFILE={RUN_PROFILE}')

RUN_TAG = f'{RUN_PROFILE}_{RUN_VARIANT}'
MODEL_FILENAME = f'best_model_gdinofrcnn_ncod_hum_{RUN_TAG}.ckpt'
LATEST_CKPT_FILENAME = f'latest_checkpoint_gdinofrcnn_ncod_hum_{RUN_TAG}.ckpt'
TRAIN_HISTORY_FILENAME = f'train_history_gdinofrcnn_ncod_hum_{RUN_TAG}.csv'
PREDICTIONS_CSV_FILENAME = f'test_predictions_gdinofrcnn_ncod_hum_{RUN_TAG}.csv'
METRICS_JSON_FILENAME = f'final_metrics_gdinofrcnn_ncod_hum_{RUN_TAG}.json'
BEST_ARTIFACT_NAME = f'best-model-gdinofrcnn-ncod-hum-{RUN_TAG}'
LAST_ARTIFACT_NAME = f'last-checkpoint-gdinofrcnn-ncod-hum-{RUN_TAG}'
FINAL_ARTIFACT_NAME = f'final-results-gdinofrcnn-ncod-hum-{RUN_TAG}'

FEAT_DIM = 1024  # DINOv3 frame
OBJ_DIM  = 2820  # FRCNN(2048) + DeBERTa cls(768) + bbox(4)
OBJ_BBOX_DIM = 4
print(f'\nBackbone: DINOv3 ({FEAT_DIM}-d frame) + GroundingDINO+FRCNN ({OBJ_DIM}-d obj, bbox split={OBJ_BBOX_DIM})')
print(f'Run profile: {RUN_PROFILE}')

class Config:
    # Paths
    video_feature_root = CLIP_FEATURE_PATH
    grounding_dino_path = GDINO_FEATURE_PATH
    sample_list_path = ANNOTATION_PATH
    split_dir_txt = SPLIT_DIR

    # Model architecture
    topK_frame = 16
    objs = 12
    frames = 16
    select_frames = 5
    topK_obj = 12

    frame_feat_dim = FEAT_DIM
    obj_feat_dim = OBJ_DIM
    use_grounding_dino = True

    # GDINO object encoding fixes
    obj_use_bbox_pos_embed = True
    obj_bbox_dim = OBJ_BBOX_DIM
    obj_hard_gather_from_frame = True

    d_model = 768
    word_dim = 768
    nheads = 8
    num_encoder_layers = 2
    num_decoder_layers = 2
    normalize_before = True
    activation = 'gelu'
    dropout = 0.3
    encoder_dropout = 0.3

    # Text encoder
    text_encoder_type = 'microsoft/deberta-base'
    freeze_text_encoder = False
    text_encoder_lr = 5e-6
    text_pool_mode = 1
    use_lora = False
    lora_r = 8
    lora_alpha = 16
    lora_dropout = 0.1
    lora_target_modules = ['query_proj', 'key_proj', 'value_proj']

    # Training
    bs = 32
    accumulation_steps = 1
    lr = 1e-5
    epoch = 10
    gpu = 0
    gamma = 0.5
    decay = 1e-4
    n_query = 5
    lambda_verifier = 0.3
    lambda_knowledge = 0.0
    return_family_id = True

    # LR scheduler + early stopping
    lr_schedule = 'cosine_warmup'
    warmup_epochs = 1
    lr_patience = 1
    min_lr = 1e-7
    early_stop_patience = 4
    early_stop_min_delta = 0.05
    early_stop_start_epoch = 5

    # Generic training improvements
    use_hard_neg_mining = True
    hard_neg_weight_max = 1.5
    use_ema = True
    ema_decay = 0.999

    # NCOD + HUM
    ncod_u_lr = 0.1
    hum_alpha = 1.0

    # Aux warmup
    aux_warmup_epochs = 2

    # Other
    hard_eval = False
    pos_ratio = 1.0
    neg_ratio = 1.0
    a = 1.0
    num_workers = 4

for _k, _v in RUN_PROFILES[RUN_PROFILE].items():
    setattr(Config, _k, _v)

if RUN_PROFILE == 'run3':
    if RUN3_REG_MODE == 'dropout':
        Config.dropout = 0.25
        Config.encoder_dropout = 0.25
    elif RUN3_REG_MODE == 'decay':
        Config.decay = 8e-5
    else:
        raise ValueError(f'Invalid RUN3_REG_MODE={RUN3_REG_MODE}')

args = Config()

if args.text_encoder_type != 'microsoft/deberta-base':
    raise ValueError('Train notebook uses DeBERTa v1 only.')

set_gpu_devices(args.gpu)
set_seed(999)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
print(f'Run tag: {RUN_TAG}')
print(f'Config: frame_feat_dim={args.frame_feat_dim}, obj_feat_dim={args.obj_feat_dim}, objs={args.objs}, select_frames={args.select_frames}')
print(f'use_grounding_dino={args.use_grounding_dino} -> obj_sorter SKIPPED')
print(f'obj_use_bbox_pos_embed={args.obj_use_bbox_pos_embed} | obj_hard_gather_from_frame={args.obj_hard_gather_from_frame}')
print(f'Effective bs: physical={args.bs} x accum={args.accumulation_steps} = {args.bs * args.accumulation_steps}')
print(f'lr={args.lr} | text_encoder_lr={args.text_encoder_lr} | decay={args.decay}')
print(f'use_lora={args.use_lora} | hard_neg={args.use_hard_neg_mining} (max={args.hard_neg_weight_max}) | ema={args.use_ema} (decay={args.ema_decay})')
print(f'lr_schedule={args.lr_schedule} | warmup_epochs={args.warmup_epochs}')
print(f'aux_warmup_epochs={args.aux_warmup_epochs} | verifier={args.lambda_verifier} | knowledge={args.lambda_knowledge}')
print(f'Early stop: start={args.early_stop_start_epoch}, patience={args.early_stop_patience}, min_delta={args.early_stop_min_delta}')
print(f'Model file: {MODEL_FILENAME}')

In [ ]:
# CELL 6: Create Datasets
print('=== CELL 6: Datasets ===')

train_ds = VideoQADataset(
    split='train', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    grounding_dino_path=args.grounding_dino_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True, return_family_id=args.return_family_id
)
val_ds = VideoQADataset(
    split='val', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    grounding_dino_path=args.grounding_dino_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True, return_family_id=args.return_family_id
)
test_ds = VideoQADataset(
    split='test', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    grounding_dino_path=args.grounding_dino_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True, return_family_id=args.return_family_id
)

# Fail before training if path resolution dropped any split.
for _split_name, _dataset in [('train', train_ds), ('val', val_ds), ('test', test_ds)]:
    if len(_dataset) == 0:
        raise RuntimeError(
            f'{_split_name} dataset is empty. Expected DINOv3 train/valid/test features '
            f'in the merged directory: {args.video_feature_root}'
        )

# Fail fast if the GDINO+FRCNN loader silently returned empty object evidence.
_probe_obj = train_ds[0][1]
_expected_shape = (args.topK_frame, args.objs, args.obj_feat_dim)
if tuple(_probe_obj.shape) != _expected_shape:
    raise RuntimeError(f'Object feature shape mismatch: got {tuple(_probe_obj.shape)}, expected {_expected_shape}')
_probe_roi = _probe_obj[..., :2048]
_probe_cls = _probe_obj[..., 2048:2816]
_probe_box = _probe_obj[..., 2816:2820]
_valid_obj = (_probe_roi.abs().sum(-1) + _probe_cls.abs().sum(-1) + _probe_box.abs().sum(-1)) > 0
if not _valid_obj.any():
    raise RuntimeError('GDINO+FRCNN object sample is entirely zero; check schema/path before training.')
print(
    'Object probe OK:', tuple(_probe_obj.shape),
    f'valid_slots={_valid_obj.float().mean().item():.3f}',
    f'roi_l2={_probe_roi[_valid_obj].norm(dim=-1).mean().item():.3f}',
    f'class_l2={_probe_cls[_valid_obj].norm(dim=-1).mean().item():.3f}',
    f'bbox_l2={_probe_box[_valid_obj].norm(dim=-1).mean().item():.3f}'
)
del _probe_obj, _probe_roi, _probe_cls, _probe_box, _valid_obj

train_loader = DataLoader(
    train_ds, args.bs, shuffle=True, num_workers=args.num_workers,
    pin_memory=torch.cuda.is_available(), persistent_workers=(args.num_workers > 0)
)
# Evaluation creates many short-lived iterators (validation and ablations).
# num_workers=0 is the reliable choice in Colab/Jupyter and prevents parent-PID,
# bad-file-descriptor and semaphore cleanup errors.
val_loader = DataLoader(
    val_ds, args.bs, shuffle=False, num_workers=0,
    pin_memory=torch.cuda.is_available()
)
test_loader = DataLoader(
    test_ds, args.bs, shuffle=False, num_workers=0,
    pin_memory=torch.cuda.is_available()
)

train_sample_keys = [f"{row.video_id}_{row.type}" for row in train_ds.sample_list.itertuples(index=False)]
train_key_to_idx = {k: i for i, k in enumerate(train_sample_keys)}

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')


In [ ]:
# CELL 7: Model + Optimizers + NCOD U + Generic Improvements
print('=== CELL 7: Model ===')
cfg = {k: v for k, v in Config.__dict__.items() if not k.startswith('_')}
cfg['device'] = device
cfg['topK_frame'] = args.select_frames
model = VideoQAmodel(**cfg)
model.to(device)

ema_model = None
if args.use_ema:
    ema_model = copy.deepcopy(model).to(device)
    ema_model.eval()
    for p in ema_model.parameters():
        p.requires_grad_(False)

non_text_params = []
text_base_params = []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if 'text_encoder' in name:
        text_base_params.append(p)
    else:
        non_text_params.append(p)

param_groups = []
if len(non_text_params) > 0:
    param_groups.append({'params': non_text_params, 'lr': args.lr, 'weight_decay': args.decay})
if len(text_base_params) > 0:
    param_groups.append({'params': text_base_params, 'lr': args.text_encoder_lr, 'weight_decay': args.decay})
if len(param_groups) == 0:
    raise RuntimeError('No trainable parameters found for optimizer_model.')

optimizer_model = torch.optim.AdamW(param_groups)

if args.lr_schedule == 'cosine_warmup':
    steps_per_epoch = max(1, math.ceil(len(train_loader) / args.accumulation_steps))
    total_steps = max(1, args.epoch * steps_per_epoch)
    warmup_steps = max(1, args.warmup_epochs * steps_per_epoch)

    def _lr_lambda(step):
        if step < warmup_steps:
            return float(step + 1) / float(warmup_steps)
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        cosine = 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
        min_ratio = args.min_lr / max(args.lr, 1e-12)
        return max(min_ratio, cosine)

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer_model, lr_lambda=_lr_lambda)
else:
    scheduler = ReduceLROnPlateau(
        optimizer_model,
        mode='max',
        factor=args.gamma,
        patience=args.lr_patience,
        threshold=args.early_stop_min_delta,
        threshold_mode='abs',
        min_lr=args.min_lr
    )

U = torch.nn.Parameter(torch.full((len(train_ds),), 1e-8, dtype=torch.float32, device=device))
optimizer_u = torch.optim.SGD([U], lr=args.ncod_u_lr)

xe = nn.CrossEntropyLoss()
bce = nn.BCEWithLogitsLoss()

save_path = os.path.join(MODEL_DIR, MODEL_FILENAME)

print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M')
print(f'Text-encoder trainable params: {sum(p.numel() for p in text_base_params)/1e6:.3f}M')
print(f'EMA enabled: {ema_model is not None}')
print(f'Scheduler: {args.lr_schedule}')
print(f'U shape: {tuple(U.shape)}')
print(f'Artifacts: best={BEST_ARTIFACT_NAME} | latest={LAST_ARTIFACT_NAME}')

In [ ]:
# CELL 8: Init W&B + Resume Checkpoint
print('=== CELL 8: Initialize W&B Run ===')

start_epoch = 1
best_acc = 0.0
best_epoch = 0
epochs_without_improvement = 0
history = {'train_loss': [], 'train_acc': [], 'val_acc': []}
history_rows = []

LATEST_CKPT_PATH = os.path.join(MODEL_DIR, LATEST_CKPT_FILENAME)
TRAIN_HISTORY_CSV_PATH = os.path.join(MODEL_DIR, TRAIN_HISTORY_FILENAME)

RESUME_FROM_CHECKPOINT = True
RESUME_SOURCE = 'wandb'
RESUME_ARTIFACT_ALIAS = 'latest'
LOCAL_RESUME_PATH = LATEST_CKPT_PATH

wandb_config = {
    'run_tag': RUN_TAG,
    'run_profile': RUN_PROFILE,
    'run_variant': RUN_VARIANT,
    'backbone': 'dinov3+groundingdino',
    'text_encoder': args.text_encoder_type,
    'use_lora': args.use_lora,
    'lora_r': args.lora_r,
    'lora_alpha': args.lora_alpha,
    'lora_dropout': args.lora_dropout,
    'full_text_finetune': not args.freeze_text_encoder,
    'physical_batch_size': args.bs,
    'accumulation_steps': args.accumulation_steps,
    'effective_batch_size': args.bs * args.accumulation_steps,
    'epochs': args.epoch,
    'lambda_verifier': args.lambda_verifier,
    'lambda_knowledge': args.lambda_knowledge,
    'ncod_u_lr': args.ncod_u_lr,
    'hum_alpha': args.hum_alpha,
    'lr_main': args.lr,
    'lr_text_encoder': args.text_encoder_lr,
    'lr_schedule': args.lr_schedule,
    'warmup_epochs': args.warmup_epochs,
    'lr_scheduler_factor': args.gamma,
    'lr_scheduler_patience': args.lr_patience,
    'min_lr': args.min_lr,
    'use_hard_neg_mining': args.use_hard_neg_mining,
    'hard_neg_weight_max': args.hard_neg_weight_max,
    'use_ema': args.use_ema,
    'ema_decay': args.ema_decay,
    'validation_metric': 'Acc_ALL_like_test',
    'early_stop_patience': args.early_stop_patience,
    'early_stop_min_delta': args.early_stop_min_delta,
    'early_stop_start_epoch': args.early_stop_start_epoch,
    'resume_enabled': RESUME_FROM_CHECKPOINT,
    'resume_source': RESUME_SOURCE,
    'platform': 'kaggle'
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name=RUN_TAG, config=wandb_config, reinit=True)
wandb.watch(model, log='gradients', log_freq=100)
print(f'W&B run: {run.url if run else "(offline)"}')

def _load_resume_checkpoint(path, map_location):
    if not os.path.exists(path):
        raise FileNotFoundError(f'Checkpoint not found: {path}')
    return torch.load(path, map_location=map_location)

if RESUME_FROM_CHECKPOINT:
    print(f'Resume enabled from: {RESUME_SOURCE}')
    try:
        checkpoint = None
        resume_path = None

        if RESUME_SOURCE == 'local':
            resume_path = LOCAL_RESUME_PATH
            checkpoint = _load_resume_checkpoint(resume_path, device)
        elif RESUME_SOURCE == 'wandb':
            api = wandb.Api()
            resume_entity = WANDB_ENTITY or api.default_entity
            artifact_path = f'{resume_entity}/{WANDB_PROJECT}/{LAST_ARTIFACT_NAME}:{RESUME_ARTIFACT_ALIAS}'
            print(f'Downloading artifact: {artifact_path}')
            artifact = api.artifact(artifact_path)
            artifact_dir = artifact.download()
            candidate_path = os.path.join(artifact_dir, LATEST_CKPT_FILENAME)
            if os.path.exists(candidate_path):
                resume_path = candidate_path
            else:
                ckpt_files = [f for f in os.listdir(artifact_dir) if f.endswith('.ckpt')]
                if not ckpt_files:
                    raise FileNotFoundError(f'No .ckpt found in artifact folder: {artifact_dir}')
                resume_path = os.path.join(artifact_dir, ckpt_files[0])
            checkpoint = _load_resume_checkpoint(resume_path, device)
        else:
            raise ValueError("RESUME_SOURCE must be 'local' or 'wandb'")

        ckpt_state = checkpoint['model_state_dict']
        model_state = model.state_dict()
        filtered_state = {}
        skipped_keys = []
        for k, v in ckpt_state.items():
            if k in model_state and v.shape != model_state[k].shape:
                skipped_keys.append(f'{k}: ckpt={list(v.shape)} vs model={list(model_state[k].shape)}')
            else:
                filtered_state[k] = v

        if skipped_keys:
            print(f'Skipped {len(skipped_keys)} keys due to shape mismatch')

        missing, unexpected = model.load_state_dict(filtered_state, strict=False)
        if missing:
            print(f'Warning: missing model keys when resume: {len(missing)}')
        if unexpected:
            print(f'Warning: unexpected model keys when resume: {len(unexpected)}')

        if not skipped_keys:
            if 'optimizer_model_state_dict' in checkpoint:
                optimizer_model.load_state_dict(checkpoint['optimizer_model_state_dict'])
            if 'optimizer_u_state_dict' in checkpoint:
                optimizer_u.load_state_dict(checkpoint['optimizer_u_state_dict'])
            if 'scheduler_state_dict' in checkpoint:
                scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            if ema_model is not None and checkpoint.get('ema_model_state_dict') is not None:
                ema_model.load_state_dict(checkpoint['ema_model_state_dict'], strict=False)
            if 'U' in checkpoint:
                with torch.no_grad():
                    u_ckpt = checkpoint['U'].to(device).float().view(-1)
                    n = min(u_ckpt.numel(), U.numel())
                    U[:n].copy_(u_ckpt[:n])

            best_acc = float(checkpoint.get('best_acc', 0.0))
            start_epoch = int(checkpoint.get('epoch', 0)) + 1
            best_epoch = int(checkpoint.get('best_epoch', 0))
            epochs_without_improvement = int(checkpoint.get('epochs_without_improvement', 0))
            history = checkpoint.get('history', history)
            history_rows = checkpoint.get('history_rows', history_rows)

            if os.path.exists(TRAIN_HISTORY_CSV_PATH):
                try:
                    history_rows = pd.read_csv(TRAIN_HISTORY_CSV_PATH).to_dict('records')
                    print(f'Loaded history CSV with {len(history_rows)} rows')
                except Exception as csv_err:
                    print(f'Warning: failed to load history CSV: {csv_err}')
        else:
            print('Optimizer/scheduler/U NOT restored due to architecture change.')

        print(f'Resumed from: {resume_path}')
        print(f'Start epoch: {start_epoch} | Best acc: {best_acc:.2f}% | Best epoch: {best_epoch}')
    except Exception as e:
        print(f'Warning: resume failed, starting from scratch. Error: {e}')
else:
    print('Resume disabled. Training starts from epoch 1.')

In [ ]:
# CELL 8: Init W&B + Resume Checkpoint
print('=== CELL 8: Initialize W&B Run ===')

start_epoch = 1
best_acc = 0.0
best_epoch = 0
epochs_without_improvement = 0
history = {'train_loss': [], 'train_acc': [], 'val_acc': []}
history_rows = []

LATEST_CKPT_PATH = os.path.join(MODEL_DIR, LATEST_CKPT_FILENAME)
TRAIN_HISTORY_CSV_PATH = os.path.join(MODEL_DIR, TRAIN_HISTORY_FILENAME)

RESUME_FROM_CHECKPOINT = True
RESUME_SOURCE = 'wandb'
RESUME_ARTIFACT_ALIAS = 'latest'
LOCAL_RESUME_PATH = LATEST_CKPT_PATH

wandb_config = {
    'run_tag': RUN_TAG,
    'run_profile': RUN_PROFILE,
    'run_variant': RUN_VARIANT,
    'backbone': 'dinov3+groundingdino',
    'text_encoder': args.text_encoder_type,
    'use_lora': args.use_lora,
    'lora_r': args.lora_r,
    'lora_alpha': args.lora_alpha,
    'lora_dropout': args.lora_dropout,
    'full_text_finetune': not args.freeze_text_encoder,
    'physical_batch_size': args.bs,
    'accumulation_steps': args.accumulation_steps,
    'effective_batch_size': args.bs * args.accumulation_steps,
    'epochs': args.epoch,
    'lambda_verifier': args.lambda_verifier,
    'lambda_knowledge': args.lambda_knowledge,
    'ncod_u_lr': args.ncod_u_lr,
    'hum_alpha': args.hum_alpha,
    'lr_main': args.lr,
    'lr_text_encoder': args.text_encoder_lr,
    'lr_schedule': args.lr_schedule,
    'warmup_epochs': args.warmup_epochs,
    'lr_scheduler_factor': args.gamma,
    'lr_scheduler_patience': args.lr_patience,
    'min_lr': args.min_lr,
    'use_hard_neg_mining': args.use_hard_neg_mining,
    'hard_neg_weight_max': args.hard_neg_weight_max,
    'use_ema': args.use_ema,
    'ema_decay': args.ema_decay,
    'validation_metric': 'Acc_ALL_like_test',
    'early_stop_patience': args.early_stop_patience,
    'early_stop_min_delta': args.early_stop_min_delta,
    'early_stop_start_epoch': args.early_stop_start_epoch,
    'resume_enabled': RESUME_FROM_CHECKPOINT,
    'resume_source': RESUME_SOURCE,
    'platform': 'kaggle'
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name=RUN_TAG, config=wandb_config, reinit=True)
wandb.watch(model, log='gradients', log_freq=100)
print(f'W&B run: {run.url if run else "(offline)"}')

def _load_resume_checkpoint(path, map_location):
    if not os.path.exists(path):
        raise FileNotFoundError(f'Checkpoint not found: {path}')
    return torch.load(path, map_location=map_location)

if RESUME_FROM_CHECKPOINT:
    print(f'Resume enabled from: {RESUME_SOURCE}')
    try:
        checkpoint = None
        resume_path = None

        if RESUME_SOURCE == 'local':
            resume_path = LOCAL_RESUME_PATH
            checkpoint = _load_resume_checkpoint(resume_path, device)
        elif RESUME_SOURCE == 'wandb':
            api = wandb.Api()
            resume_entity = WANDB_ENTITY or api.default_entity
            artifact_path = f'{resume_entity}/{WANDB_PROJECT}/{LAST_ARTIFACT_NAME}:{RESUME_ARTIFACT_ALIAS}'
            print(f'Downloading artifact: {artifact_path}')
            artifact = api.artifact(artifact_path)
            artifact_dir = artifact.download()
            candidate_path = os.path.join(artifact_dir, LATEST_CKPT_FILENAME)
            if os.path.exists(candidate_path):
                resume_path = candidate_path
            else:
                ckpt_files = [f for f in os.listdir(artifact_dir) if f.endswith('.ckpt')]
                if not ckpt_files:
                    raise FileNotFoundError(f'No .ckpt found in artifact folder: {artifact_dir}')
                resume_path = os.path.join(artifact_dir, ckpt_files[0])
            checkpoint = _load_resume_checkpoint(resume_path, device)
        else:
            raise ValueError("RESUME_SOURCE must be 'local' or 'wandb'")

        ckpt_state = checkpoint['model_state_dict']
        model_state = model.state_dict()
        filtered_state = {}
        skipped_keys = []
        for k, v in ckpt_state.items():
            if k in model_state and v.shape != model_state[k].shape:
                skipped_keys.append(f'{k}: ckpt={list(v.shape)} vs model={list(model_state[k].shape)}')
            else:
                filtered_state[k] = v

        if skipped_keys:
            print(f'Skipped {len(skipped_keys)} keys due to shape mismatch')

        missing, unexpected = model.load_state_dict(filtered_state, strict=False)
        if missing:
            print(f'Warning: missing model keys when resume: {len(missing)}')
        if unexpected:
            print(f'Warning: unexpected model keys when resume: {len(unexpected)}')

        if not skipped_keys:
            if 'optimizer_model_state_dict' in checkpoint:
                optimizer_model.load_state_dict(checkpoint['optimizer_model_state_dict'])
            if 'optimizer_u_state_dict' in checkpoint:
                optimizer_u.load_state_dict(checkpoint['optimizer_u_state_dict'])
            if 'scheduler_state_dict' in checkpoint:
                scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            if ema_model is not None and checkpoint.get('ema_model_state_dict') is not None:
                ema_model.load_state_dict(checkpoint['ema_model_state_dict'], strict=False)
            if 'U' in checkpoint:
                with torch.no_grad():
                    u_ckpt = checkpoint['U'].to(device).float().view(-1)
                    n = min(u_ckpt.numel(), U.numel())
                    U[:n].copy_(u_ckpt[:n])

            best_acc = float(checkpoint.get('best_acc', 0.0))
            start_epoch = int(checkpoint.get('epoch', 0)) + 1
            best_epoch = int(checkpoint.get('best_epoch', 0))
            epochs_without_improvement = int(checkpoint.get('epochs_without_improvement', 0))
            history = checkpoint.get('history', history)
            history_rows = checkpoint.get('history_rows', history_rows)

            if os.path.exists(TRAIN_HISTORY_CSV_PATH):
                try:
                    history_rows = pd.read_csv(TRAIN_HISTORY_CSV_PATH).to_dict('records')
                    print(f'Loaded history CSV with {len(history_rows)} rows')
                except Exception as csv_err:
                    print(f'Warning: failed to load history CSV: {csv_err}')
        else:
            print('Optimizer/scheduler/U NOT restored due to architecture change.')

        print(f'Resumed from: {resume_path}')
        print(f'Start epoch: {start_epoch} | Best acc: {best_acc:.2f}% | Best epoch: {best_epoch}')
    except Exception as e:
        print(f'Warning: resume failed, starting from scratch. Error: {e}')
else:
    print('Resume disabled. Training starts from epoch 1.')

In [ ]:
# CELL 10: Detailed Evaluation + CSV export (no knowledge bank / memory gating)
print('=== CELL 10: Detailed Evaluation (Model Only) ===')

CSV_OUTPUT_PATH = os.path.join(MODEL_DIR, PREDICTIONS_CSV_FILENAME)
COMPARISON_CSV_PATH = os.path.join(MODEL_DIR, 'run_comparison_gdino_3run.csv')

def _build_eval_meta_map(loader):
    sample_list = getattr(getattr(loader, 'dataset', None), 'sample_list', None)
    meta_map = {}
    if sample_list is None:
        return meta_map
    for _, row in sample_list.iterrows():
        vid = str(row.get('video_id', ''))
        qtype = str(row.get('type', 'unknown'))
        meta_map[f'{vid}_{qtype}'] = {
            'video_id': vid,
            'question_type': qtype,
            'question': str(row.get('question', '')),
            'answers': [str(row.get(f'a{i}', '')) for i in range(5)]
        }
    return meta_map

def evaluate_detailed_v2(model, loader, device, log_to_wandb=True):
    model.eval()
    type_results, prediction_rows = {}, []
    meta_map = _build_eval_meta_map(loader)
    with torch.no_grad():
        for batch in tqdm(loader):
            ff, of, qns, ans_word, ans_id, qns_keys, q_family_id = _unpack_batch(batch)
            ff, of = ff.to(device), of.to(device)
            q_family_id = q_family_id.to(device) if q_family_id is not None else None
            out = model(ff, of, qns, ans_word, return_aux=True, q_family_id=q_family_id)
            logits = out.get('fused_score', out['logits'])
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            preds, targets = probs.argmax(axis=-1), ans_id.numpy()

            for i, (key, pred, target, prob_vec) in enumerate(zip(qns_keys, preds, targets, probs)):
                meta = meta_map.get(str(key), {})
                qtype = str(meta.get('question_type', 'unknown'))
                video_id = str(meta.get('video_id', str(key)))
                answers = list(meta.get('answers', ['', '', '', '', '']))[:5]
                answers += [''] * (5 - len(answers))
                correct_idx, predicted_idx = int(target), int(pred)
                is_correct = int(correct_idx == predicted_idx)
                type_results.setdefault(qtype, []).append({
                    'video_id': video_id, 'pred': predicted_idx,
                    'target': correct_idx, 'correct': bool(is_correct)
                })
                row = {
                    'video_id': video_id, 'question_type': qtype,
                    'question': str(meta.get('question', qns[i])),
                    'correct_idx': correct_idx, 'predicted_idx': predicted_idx,
                    'is_correct': is_correct,
                    'confidence': float(prob_vec[predicted_idx]),
                    'predicted_answer': answers[predicted_idx],
                    'correct_answer': answers[correct_idx]
                }
                for j in range(5):
                    row[f'a{j}'] = answers[j]
                    row[f'prob_a{j}'] = float(prob_vec[j])
                prediction_rows.append(row)

    if not prediction_rows:
        raise RuntimeError(
            'Detailed evaluation produced zero predictions. Check that test_ds/test_loader is non-empty.'
        )
    prediction_df = pd.DataFrame(prediction_rows)
    prediction_df.to_csv(CSV_OUTPUT_PATH, index=False)
    print(f'Saved detailed predictions CSV: {CSV_OUTPUT_PATH}')

    metrics = {}
    mapping = [
        ('Description', 'descriptive'), ('Explanation', 'explanatory'),
        ('Predictive-Answer', 'predictive'), ('Predictive-Reason', 'predictive_reason'),
        ('Counterfactual-Answer', 'counterfactual'),
        ('Counterfactual-Reason', 'counterfactual_reason')
    ]
    for name, qtype in mapping:
        rows = type_results.get(qtype, [])
        metrics[name] = 100.0 * sum(r['correct'] for r in rows) / max(len(rows), 1)

    def _paired_metric(ans_type, reason_type):
        ans = {r['video_id']: r['correct'] for r in type_results.get(ans_type, [])}
        reason = {r['video_id']: r['correct'] for r in type_results.get(reason_type, [])}
        common = set(ans) & set(reason)
        return 100.0 * sum(ans[v] and reason[v] for v in common) / max(len(common), 1)

    metrics['PAR'] = _paired_metric('predictive', 'predictive_reason')
    metrics['CAR'] = _paired_metric('counterfactual', 'counterfactual_reason')
    metrics['Acc_ALL'] = (metrics['Description'] + metrics['Explanation'] + metrics['PAR'] + metrics['CAR']) / 4.0
    metrics['WeightedScore_WeakPriority'] = (
        0.35 * metrics['Predictive-Reason'] +
        0.35 * metrics['Counterfactual-Reason'] +
        0.20 * metrics['Acc_ALL'] + 0.10 * float(best_acc)
    )
    if log_to_wandb and wandb.run is not None:
        wandb.log({f'eval/{k.replace("-", "_")}': float(v) for k, v in metrics.items()})
    print(f"PAR: {metrics['PAR']:.2f}% | CAR: {metrics['CAR']:.2f}% | Acc_ALL: {metrics['Acc_ALL']:.2f}%")
    return metrics, type_results, CSV_OUTPUT_PATH

metrics, raw_results, predictions_csv = evaluate_detailed_v2(model, test_loader, device, log_to_wandb=True)
comparison_row = {
    'run_tag': RUN_TAG, 'run_profile': RUN_PROFILE,
    'best_val_acc': float(best_acc), 'best_epoch': int(best_epoch),
    **{k: float(v) for k, v in metrics.items()}
}
if os.path.exists(COMPARISON_CSV_PATH):
    comp_df = pd.read_csv(COMPARISON_CSV_PATH)
    comp_df = comp_df[comp_df['run_tag'] != RUN_TAG]
    comp_df = pd.concat([comp_df, pd.DataFrame([comparison_row])], ignore_index=True)
else:
    comp_df = pd.DataFrame([comparison_row])
comp_df = comp_df.sort_values('WeightedScore_WeakPriority', ascending=False)
comp_df.to_csv(COMPARISON_CSV_PATH, index=False)
display(comp_df[['run_tag', 'best_val_acc', 'Predictive-Reason', 'Counterfactual-Reason', 'Acc_ALL', 'WeightedScore_WeakPriority']])
if wandb.run is not None:
    wandb.run.summary.update({
        'run_tag': RUN_TAG, 'run_profile': RUN_PROFILE,
        'weighted_score_weak_priority': float(metrics['WeightedScore_WeakPriority'])
    })
print(metrics)


In [ ]:
# CELL 10B: Object branch diagnostics + ROI/Class/BBox ablation
print('=== CELL 10B: Object Branch Diagnostics and Ablation ===')

ROI_DIM, CLASS_DIM, BBOX_DIM = 2048, 768, 4
assert args.obj_feat_dim == ROI_DIM + CLASS_DIM + BBOX_DIM

def _branch_statistics(model, loader, device, max_batches=50):
    sums = {k: 0.0 for k in [
        'roi_input_l2', 'class_input_l2', 'bbox_input_l2',
        'roi_projected_l2', 'class_projected_l2', 'bbox_projected_l2',
        'valid_object_ratio'
    ]}
    count = 0
    model.eval()
    with torch.no_grad():
        for bi, batch in enumerate(tqdm(loader, desc='Object diagnostics')):
            if bi >= max_batches:
                break
            _, of, *_ = _unpack_batch(batch)
            of = of.to(device)
            roi = of[..., :ROI_DIM]
            cls = of[..., ROI_DIM:ROI_DIM + CLASS_DIM]
            bbox = of[..., -BBOX_DIM:]
            valid = (roi.abs().sum(-1) + cls.abs().sum(-1) + bbox.abs().sum(-1)) > 0
            if not valid.any():
                continue

            roi_v, cls_v, bbox_v = roi[valid], cls[valid], bbox[valid]
            sem_norm = model.obj_sem_norm(torch.cat([roi_v, cls_v], dim=-1))
            w = model.obj_resize.fc.weight
            roi_proj = F.linear(sem_norm[..., :ROI_DIM], w[:, :ROI_DIM], None)
            cls_proj = F.linear(sem_norm[..., ROI_DIM:], w[:, ROI_DIM:], None)
            bbox_proj = model.bbox_pos_embed(bbox_v)

            sums['roi_input_l2'] += roi_v.norm(dim=-1).mean().item()
            sums['class_input_l2'] += cls_v.norm(dim=-1).mean().item()
            sums['bbox_input_l2'] += bbox_v.norm(dim=-1).mean().item()
            sums['roi_projected_l2'] += roi_proj.norm(dim=-1).mean().item()
            sums['class_projected_l2'] += cls_proj.norm(dim=-1).mean().item()
            sums['bbox_projected_l2'] += bbox_proj.norm(dim=-1).mean().item()
            sums['valid_object_ratio'] += valid.float().mean().item()
            count += 1

    stats = {k: v / max(count, 1) for k, v in sums.items()}
    stats['projected_roi_to_class_ratio'] = stats['roi_projected_l2'] / max(stats['class_projected_l2'], 1e-8)
    stats['input_roi_to_class_ratio'] = stats['roi_input_l2'] / max(stats['class_input_l2'], 1e-8)
    return stats

def _apply_object_ablation(of, mode):
    x = of.clone()
    if mode == 'normal':
        return x
    if mode == 'no_roi':
        x[..., :ROI_DIM] = 0
    elif mode == 'no_class':
        x[..., ROI_DIM:ROI_DIM + CLASS_DIM] = 0
    elif mode == 'no_bbox':
        x[..., -BBOX_DIM:] = 0
    elif mode == 'roi_only':
        x[..., ROI_DIM:] = 0
    elif mode == 'class_only':
        x[..., :ROI_DIM] = 0
        x[..., -BBOX_DIM:] = 0
    elif mode == 'zero_object':
        x.zero_()
    else:
        raise ValueError(mode)
    return x

def _eval_object_ablation(model, loader, device, mode):
    model.eval()
    correct, total = 0, 0
    type_results = {}
    with torch.no_grad():
        for batch in tqdm(loader, desc=f'Ablation {mode}', leave=False):
            ff, of, q, a, ans_id, qns_keys, q_family_id = _unpack_batch(batch)
            ff, tgt = ff.to(device), ans_id.to(device)
            of = _apply_object_ablation(of.to(device), mode)
            q_family_id = q_family_id.to(device) if q_family_id is not None else None
            out = model(ff, of, q, a, return_aux=True, q_family_id=q_family_id)
            logits = out.get('fused_score', out['logits'])
            preds = logits.argmax(-1)
            correct += (preds == tgt).sum().item()
            total += tgt.numel()
            for key, pred, target in zip(qns_keys, preds.cpu().tolist(), tgt.cpu().tolist()):
                video_id, qtype = _split_qns_key(key)
                type_results.setdefault(qtype, []).append({
                    'video_id': video_id, 'correct': bool(pred == target)
                })
    m = _compute_acc_all_metrics(type_results)
    m['Plain_Acc'] = 100.0 * correct / max(total, 1)
    return m

if len(val_loader.dataset) == 0:
    raise RuntimeError('Cannot run object diagnostics: validation dataset is empty.')
branch_stats = _branch_statistics(model, val_loader, device)
if branch_stats['valid_object_ratio'] <= 0:
    raise RuntimeError('Object diagnostics found no valid object features in validation data.')
print(pd.Series(branch_stats).round(4).to_string())

# Hard Top-K makes every ablation use exactly the same selected frames.
_old_hard_eval = model.hard_eval
model.hard_eval = True
ablation_modes = ['normal', 'no_roi', 'no_class', 'no_bbox', 'roi_only', 'class_only', 'zero_object']
ablation_rows = []
for mode in ablation_modes:
    m = _eval_object_ablation(model, val_loader, device, mode)
    ablation_rows.append({'mode': mode, **m})
model.hard_eval = _old_hard_eval

object_ablation_df = pd.DataFrame(ablation_rows)
normal_acc = float(object_ablation_df.loc[object_ablation_df['mode'] == 'normal', 'Acc_ALL'].iloc[0])
object_ablation_df['drop_vs_normal_Acc_ALL'] = normal_acc - object_ablation_df['Acc_ALL']
OBJECT_ABLATION_CSV_PATH = os.path.join(MODEL_DIR, f'object_branch_ablation_{RUN_TAG}.csv')
object_ablation_df.to_csv(OBJECT_ABLATION_CSV_PATH, index=False)
display(object_ablation_df[['mode', 'Acc_ALL', 'Plain_Acc', 'PAR', 'CAR', 'drop_vs_normal_Acc_ALL']])
print(f'Saved: {OBJECT_ABLATION_CSV_PATH}')

if wandb.run is not None:
    wandb.log({f'object_ablation/{r["mode"]}_Acc_ALL': float(r['Acc_ALL']) for r in ablation_rows})
    wandb.log({f'object_diagnostics/{k}': float(v) for k, v in branch_stats.items()})


In [ ]:
# CELL 10C: Balanced qualitative analysis of 10 wrong predictions
print('=== CELL 10C: Ten Wrong Prediction Examples ===')
if not os.path.exists(predictions_csv) or os.path.getsize(predictions_csv) == 0:
    raise RuntimeError('Prediction CSV is missing or empty; run CELL 10 successfully first.')
prediction_df = pd.read_csv(predictions_csv)
if prediction_df.empty:
    raise RuntimeError('Prediction CSV has no rows; check test dataset resolution.')
wrong_df = prediction_df[prediction_df['is_correct'] == 0].copy()
if wrong_df.empty:
    print('No wrong predictions found; skipping qualitative error samples.')
wrong_df['gt_probability'] = wrong_df.apply(
    lambda r: float(r.get(f"prob_a{int(r['correct_idx'])}", 0.0)), axis=1
)
wrong_df['wrong_margin'] = wrong_df['confidence'] - wrong_df['gt_probability']
wrong_df = wrong_df.sort_values(['question_type', 'wrong_margin'], ascending=[True, False])

# Round-robin selection guarantees broad question-type coverage before adding extras.
groups = {q: g.reset_index(drop=True) for q, g in wrong_df.groupby('question_type')}
selected = []
depth = 0
while len(selected) < 10 and any(depth < len(g) for g in groups.values()):
    for qtype in sorted(groups):
        if depth < len(groups[qtype]) and len(selected) < 10:
            selected.append(groups[qtype].iloc[depth])
    depth += 1

error_samples_df = pd.DataFrame(selected)
ERROR_SAMPLES_CSV_PATH = os.path.join(MODEL_DIR, f'error_samples_10_{RUN_TAG}.csv')
error_samples_df.to_csv(ERROR_SAMPLES_CSV_PATH, index=False)

show_cols = [
    'video_id', 'question_type', 'question', 'correct_answer',
    'predicted_answer', 'confidence', 'gt_probability', 'wrong_margin'
]
display(error_samples_df[show_cols].style.format({
    'confidence': '{:.3f}', 'gt_probability': '{:.3f}', 'wrong_margin': '{:.3f}'
}))
for i, row in error_samples_df.reset_index(drop=True).iterrows():
    candidates = ' | '.join([f"{j}:{row[f'a{j}']} ({row[f'prob_a{j}']:.3f})" for j in range(5)])
    print(f"\n[{i+1}] {row['question_type']} | video={row['video_id']}")
    print('Q:', row['question'])
    print('GT:', row['correct_answer'])
    print('Pred:', row['predicted_answer'])
    print('Candidates:', candidates)
print(f'\nSaved: {ERROR_SAMPLES_CSV_PATH}')


In [ ]:
# CELL 11: Upload exactly one last checkpoint and one best checkpoint
print('=== CELL 11: Finish W&B (best + last only) ===')

METRICS_JSON_PATH = os.path.join(MODEL_DIR, METRICS_JSON_FILENAME)
with open(METRICS_JSON_PATH, 'w') as f:
    json.dump(metrics, f, indent=2)

if wandb.run is not None:
    if os.path.exists(LATEST_CKPT_PATH):
        last_artifact = wandb.Artifact(
            name=LAST_ARTIFACT_NAME, type='model',
            metadata={'checkpoint_kind': 'last', 'run_tag': RUN_TAG,
                      'epoch': int(history_rows[-1]['epoch']) if history_rows else 0}
        )
        last_artifact.add_file(LATEST_CKPT_PATH, name=LATEST_CKPT_FILENAME)
        wandb.log_artifact(last_artifact, aliases=['last'])
        print('Uploaded one last checkpoint artifact.')
    else:
        print(f'Warning: last checkpoint not found: {LATEST_CKPT_PATH}')

    if os.path.exists(save_path):
        best_artifact = wandb.Artifact(
            name=BEST_ARTIFACT_NAME, type='model',
            metadata={'checkpoint_kind': 'best', 'run_tag': RUN_TAG,
                      'best_epoch': int(best_epoch), 'best_val_acc': float(best_acc)}
        )
        best_artifact.add_file(save_path, name=MODEL_FILENAME)
        wandb.log_artifact(best_artifact, aliases=['best'])
        print('Uploaded one best checkpoint artifact.')
    else:
        print(f'Warning: best checkpoint not found: {save_path}')
    wandb.finish()
print('W&B finished: metrics history + exactly best/last checkpoint artifacts.')


In [ ]:
# CELL 12: Disconnect Colab runtime after uploads
print('Training and W&B uploads complete. Disconnecting runtime...')
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as exc:
    print(f'Automatic disconnect unavailable: {exc}')
